In [1]:
%cd ..

/home/kiwisaki/Desktop/ProjectHub/stock-gen


In [2]:
from prompt_to_video.core.prompt import generate_video_script
from prompt_to_video.core.video import generate_video_from_objects
from pydantic import BaseModel
from prompt_to_video.settings import (
    TEXT_GENERATION_MODEL,
    VIDEO_SCRIPT_GENERATION_SYSTEM_PROMPT,
)
from typing import Any
from openai import OpenAI


class ImagePrompts(BaseModel):
    """Class for image prompt, which is used to generate image."""

    prompts: list[str]


def generate_image_prompts(
    video_script: Any,
    user_prompt: str,
    client: OpenAI,
) -> ImagePrompts:
    """Function which takes video idea prompt and returns prepared video script."""
    return (
        client.beta.chat.completions.parse(
            model=TEXT_GENERATION_MODEL,
            messages=[
                {"role": "system", "content": VIDEO_SCRIPT_GENERATION_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
                {"role": "system", "content": str(video_script)},
                {
                    "role": "user",
                    "content": "Based on the video script, generate image prompts from scenes, be detailed, as it will be used as standalone prompts, and context from the video script will be lost. Each scene should correspond to one prompt.",
                },  # noqa: E501
            ],
            response_format=ImagePrompts,
        )
        .choices[0]
        .message.parsed
    )

2025-02-17 20:43:25,209 [INFO] - <prompt_to_video> - Loaded environment variables from .env file.


In [3]:
client = OpenAI()
user_query = """

"""
video_script = generate_video_script(user_query, client)
image_prompts = generate_image_prompts(
    video_script,
    "Generate image prompts from video script, each prompt itself must include style, be precise and clear what they mean.",
    client,
)  # noqa: E501
# image_prompts = [scene.description for scene in video_script.get_all_scenes()]
text_to_voice_prompt = video_script.get_full_narrative()

2025-02-17 20:43:40,406 [INFO] - <httpx> - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-02-17 20:43:56,212 [INFO] - <httpx> - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
from prompt_to_video.core.image import ImageGenerator

image_generator = ImageGenerator()

In [ ]:
images = [
    image_generator.generate_image(
        image_prompt, height=1080, width=1920, num_inference_steps=4
    )
    for image_prompt in image_prompts.prompts
]

In [13]:
from prompt_to_video.core.audio.tts import generate_audio

audio = generate_audio(text_to_voice_prompt, "en-GB-RyanNeural")

In [ ]:
a = generate_video_from_objects("experimentas1", [images], [audio])